# 01 — Data Collection: World Bank Labour Indicators for Pakistan

**Project:** Forecasting Youth Unemployment in Pakistan
**Author:** Awais Ahmed Shah — BS Data Science (Final Year), UET Peshawar

## What this notebook does
1. Downloads six labour-market indicators for Pakistan (1991–2025) from the **World Bank Open Data API**.
2. Saves the raw API responses (JSON + a long-format CSV transcribed exactly from them).
3. Builds one clean, wide, analysis-ready CSV in `data/processed/`.

If the API is not reachable (offline / blocked network), the notebook falls back to the
bundled raw CSV in `data/raw/worldbank_pakistan_long.csv`, so it can always be re-run.

## Indicators (World Bank codes)
| Code | Meaning |
|---|---|
| `SL.UEM.1524.ZS` | Youth unemployment, total (% of labour force ages 15–24), modelled ILO estimate |
| `SL.UEM.1524.MA.ZS` | Youth unemployment, male |
| `SL.UEM.1524.FE.ZS` | Youth unemployment, female |
| `SL.UEM.TOTL.ZS` | Unemployment, total (% of labour force), modelled ILO estimate |
| `SL.TLF.CACT.ZS` | Labour force participation rate, total (% of population ages 15+), modelled ILO estimate |
| `NY.GDP.PCAP.CD` | GDP per capita (current US$) |

All series are *modelled ILO estimates* published by the World Bank — they are not raw survey
microdata. Pakistan's own survey figures (Pakistan Bureau of Statistics, Labour Force Survey)
are documented separately in `data/sources.md`.

In [1]:
from pathlib import Path

def find_project_root():
    """Walk up from the notebook folder until the folder containing data/ is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (folder containing data/processed).")

ROOT = find_project_root()
print("Project root:", ROOT)

Project root: /home/hatch/workspace/youth-unemployment-repo


In [2]:
import json
import urllib.request
import urllib.error
import pandas as pd

INDICATORS = {
    "SL.UEM.1524.ZS": "youth_unemployment_total",
    "SL.UEM.1524.MA.ZS": "youth_unemployment_male",
    "SL.UEM.1524.FE.ZS": "youth_unemployment_female",
    "SL.UEM.TOTL.ZS": "unemployment_total",
    "SL.TLF.CACT.ZS": "labour_force_participation_total",
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
}
START_YEAR, END_YEAR = 1991, 2025

## Step 1 — Try to download each indicator from the World Bank API

In [3]:
def fetch_indicator(code):
    url = (f"https://api.worldbank.org/v2/country/PAK/indicator/{code}"
           f"?date={START_YEAR}:{END_YEAR}&per_page=100&format=json")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode("utf-8"))

raw_dir = ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

series = {}
api_ok = True
for code in INDICATORS:
    try:
        payload = fetch_indicator(code)
        with open(raw_dir / f"wb_{code}.json", "w") as fh:
            json.dump(payload, fh)
        rows = payload[1]
        series[code] = {int(r["date"]): r["value"] for r in rows}
        print(f"OK   {code}: {len(rows)} rows downloaded")
    except Exception as exc:  # network blocked, timeout, etc.
        api_ok = False
        print(f"FAIL {code}: {exc}")

print("\nAPI download fully successful:", api_ok)

OK   SL.UEM.1524.ZS: 35 rows downloaded


OK   SL.UEM.1524.MA.ZS: 35 rows downloaded


OK   SL.UEM.1524.FE.ZS: 35 rows downloaded


OK   SL.UEM.TOTL.ZS: 35 rows downloaded


OK   SL.TLF.CACT.ZS: 35 rows downloaded


OK   NY.GDP.PCAP.CD: 35 rows downloaded

API download fully successful: True


## Step 2 — Fallback: use the bundled raw CSV if the API failed

The bundled CSV was transcribed **exactly** from API JSON responses fetched for this
project, so the numbers are identical either way.

In [4]:
if not api_ok or len(series) < len(INDICATORS):
    print("Using bundled fallback: data/raw/worldbank_pakistan_long.csv")
    long_df = pd.read_csv(raw_dir / "worldbank_pakistan_long.csv")
    series = {
        code: dict(zip(g["year"], g["value"]))
        for code, g in long_df.groupby("indicator_code")
    }

# Save a long-format raw CSV (exact values, one row per indicator-year)
long_rows = [
    {"indicator_code": code, "year": year, "value": value}
    for code, values in series.items() for year, value in sorted(values.items())
]
long_df = pd.DataFrame(long_rows).sort_values(["indicator_code", "year"])
long_df.to_csv(raw_dir / "worldbank_pakistan_long.csv", index=False)
print("Saved data/raw/worldbank_pakistan_long.csv with", len(long_df), "rows")

Saved data/raw/worldbank_pakistan_long.csv with 210 rows


## Step 3 — Build the wide, processed dataset

In [5]:
wide = pd.DataFrame({"year": range(START_YEAR, END_YEAR + 1)})
for code, column in INDICATORS.items():
    wide[column] = [series[code].get(y) for y in wide["year"]]

out_path = ROOT / "data" / "processed" / "pakistan_labour_indicators_1991_2025.csv"
wide.to_csv(out_path, index=False)
print("Saved:", out_path)
print("\nMissing values per column:")
print(wide.isna().sum())
wide

Saved: /home/hatch/workspace/youth-unemployment-repo/data/processed/pakistan_labour_indicators_1991_2025.csv

Missing values per column:
year                                0
youth_unemployment_total            0
youth_unemployment_male             0
youth_unemployment_female           0
unemployment_total                  0
labour_force_participation_total    0
gdp_per_capita_usd                  0
dtype: int64


,year,youth_unemployment_total,youth_unemployment_male,youth_unemployment_female,unemployment_total,labour_force_participation_total,gdp_per_capita_usd
0,1991,0.960,1.032,0.577,0.596,49.680,380.019843
1,1992,0.958,1.030,0.573,0.587,49.680,395.677209
2,1993,0.967,1.037,0.594,0.596,49.200,407.888141
3,1994,0.972,1.042,0.604,0.601,49.150,399.955928
4,1995,0.973,1.029,0.611,0.611,48.360,450.550279
5,1996,0.967,1.032,0.597,0.600,49.240,456.996962
6,1997,0.966,1.040,0.594,0.611,50.120,438.030943
7,1998,0.970,1.047,0.603,0.620,49.950,424.453698
8,1999,0.972,1.029,0.641,0.625,50.180,418.249257
9,2000,0.967,1.010,0.668,0.614,50.410,642.338346


## Step 4 — Sanity check on the headline number

In [6]:
row_2025 = wide.loc[wide["year"] == 2025].iloc[0]
print(f"Youth unemployment (total), Pakistan, 2025 = {row_2025['youth_unemployment_total']}%")
print("Source: World Bank API, indicator SL.UEM.1524.ZS (modelled ILO estimate).")
print("Full API URL pattern: https://api.worldbank.org/v2/country/PAK/indicator/<CODE>?date=1991:2025&per_page=100&format=json")

Youth unemployment (total), Pakistan, 2025 = 9.594%
Source: World Bank API, indicator SL.UEM.1524.ZS (modelled ILO estimate).
Full API URL pattern: https://api.worldbank.org/v2/country/PAK/indicator/<CODE>?date=1991:2025&per_page=100&format=json
